In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model="claude-haiku-4-5"


In [2]:
tools = [
    {
        "name": "lookup_order",
        "description": "Look up an order by its order ID and return its item, status, order date, and total. Call this whenever the customer references an order number or asks about the state of an existing order.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID, e.g. A1001"
                }
            },
            "required": ["order_id"]
        }
    },
    {
        # Structured output demo: this tool's only job is to shape Claude's
        # output. We mark it `strict: True` so the API guarantees the
        # returned input matches the schema exactly — every field present,
        # enums honored, no extra keys — no parsing/regex needed on our side.
        "name": "extract_return_request",
        "description": "Call this whenever a customer describes a return or refund request, to capture it as structured data before deciding what to do next. Fill in every field as best you can from the conversation; use null/'unclear' and list missing_information for anything not yet stated.",
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": ["string", "null"],
                    "description": "The order ID mentioned by the customer, e.g. A1001, or null if they haven't given one."
                },
                "reason": {
                    "type": "string",
                    "enum": [
                        "defective_item",
                        "wrong_item_shipped",
                        "changed_mind",
                        "billing_dispute",
                        "late_delivery",
                        "unclear",
                        "other"
                    ],
                    "description": "The customer's stated reason for the return/refund. Use 'unclear' if not stated clearly enough to classify."
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                    "description": "How urgent the request sounds, based on tone and content (e.g. angry, time-sensitive)."
                },
                "missing_information": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["order_id", "reason", "item_condition", "preferred_resolution"]
                    },
                    "description": "Which pieces of information the customer has not yet provided."
                },
                "summary": {
                    "type": "string",
                    "description": "One-sentence, neutral summary of what the customer wants."
                }
            },
            "required": ["order_id", "reason", "urgency", "missing_information", "summary"],
            "additionalProperties": False
        }
    }
]

In [3]:
import json
from pathlib import Path

ORDERS_FILE = Path("orders.json")


def lookup_order(order_id: str) -> str:
    """Look up an order by ID in orders.json and return its details as JSON, or an error message."""
    orders = json.loads(ORDERS_FILE.read_text())

    for order in orders:
        if order["order_id"].lower() == order_id.lower():
            return json.dumps(order)

    return f"Error: no order found with ID '{order_id}'."


def process_return_request(order_id, reason, urgency, missing_information, summary) -> str:
    """Apply business rules to an extracted return request and return a JSON
    result describing what should happen next — Claude reads this to decide
    how to phrase its reply, rather than us scripting the reply text here."""
    if not order_id or "order_id" in missing_information:
        return json.dumps({"decision": "need_order_id"})

    orders = json.loads(ORDERS_FILE.read_text())
    order = next((o for o in orders if o["order_id"].lower() == order_id.lower()), None)
    if order is None:
        return json.dumps({"decision": "order_not_found", "order_id": order_id})

    if reason == "unclear":
        decision = "need_clarification"
    elif reason == "billing_dispute" or urgency == "high":
        decision = "escalate_to_specialist"
    else:
        decision = "approve_return"

    return json.dumps({
        "decision": decision,
        "order": order,
        "reason": reason,
        "urgency": urgency,
        "summary": summary,
    })


def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Dispatch a tool_use block to its implementation."""
    if tool_name == "lookup_order":
        return lookup_order(tool_input["order_id"])

    if tool_name == "extract_return_request":
        return process_return_request(**tool_input)

    return f"Error: unknown tool '{tool_name}'."

In [4]:
def add_user_message(messages: list, text: str) -> None:
    """Append a user turn to the conversation."""
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages: list, text: str) -> None:
    """Append an assistant turn to the conversation."""
    messages.append({"role": "assistant", "content": text})

In [5]:
SYSTEM_PROMPT = """You are a helpful shop assistant for an online store.

If the customer is just asking about the status of an existing order, call
lookup_order directly.

If the customer describes wanting a return or refund, call
extract_return_request first — even if they haven't given you every detail
yet, since the tool's result will tell you what's missing and what to do
next. Fill in every field of that tool as best you can from the conversation
so far; don't ask the customer questions before calling it.

Follow the extract_return_request result's "decision" field to guide your
reply:
- need_order_id: ask for their order ID.
- order_not_found: let them know that order ID doesn't match anything, and ask them to double-check it.
- need_clarification: ask why they'd like to return the item.
- escalate_to_specialist: tell them you're escalating this to a specialist.
- approve_return: confirm the return and explain next steps.
"""

# The whole conversation lives in this list — we resend it on every call.
messages = []

def send_message(messages: list) -> str:
    """Call Claude with the current conversation, running the tool-use loop
    until Claude has no more tools to call, and return the final reply text."""
    while True:
        response = client.messages.create(
            model=model,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=tools,
            messages=messages,
        )

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return next(block.text for block in response.content if block.type == "text")

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"[tool call] {block.name}({block.input})")
                result = execute_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })

        messages.append({"role": "user", "content": tool_results})

In [6]:
while True:
    try:
        user_input = input("You: ")
        print(f"User: {user_input}")
    except (EOFError, KeyboardInterrupt):
        break

    # VS Code's input box returns "" for both Escape and a blank Enter, so
    # blank input doubles as the way to quit here.
    if not user_input.strip() or user_input.lower() in ("quit", "exit"):
        break

    add_user_message(messages, user_input)
    # send_message appends the assistant turn(s) itself, including any
    # intermediate tool_use/tool_result turns from the tool loop.
    reply = send_message(messages)

    print(f"Assistant: {reply}")

User: Hello I want to return my mouse I bought 
Assistant: I'd be happy to help you with a return for your mouse! To get started, I need to gather some information about your return request.

Could you please provide your **order ID**? It's usually something like A1001 or similar, and you can find it in your order confirmation email.
User: my order is A1001
[tool call] extract_return_request({'order_id': 'A1001', 'reason': 'unclear', 'urgency': 'medium', 'missing_information': ['reason', 'item_condition', 'preferred_resolution'], 'summary': 'Customer wants to return a mouse from order A1001.'})
Assistant: Great! I found your order. You purchased a **Wireless Mouse** for **$24.99**, which was delivered on July 10, 2026.

To process your return, could you tell me **why you'd like to return the mouse**? For example:
- Is it **defective or not working properly**?
- Did you receive the **wrong item**?
- Have you **changed your mind** about the purchase?
- Is there another reason?

This will